In [6]:
from modify import modify

orig = """count = 0
def print():
  hello()

if __name__ == '__main__':
  print(123)
  print('well')
  exit()
"""
diff = """# 定位上文
 def print():
# 增加文本
+  global count
+  count += 1
===
 if __name__ == '__main__':
-  print(123)
+  print('welcome to my world')
+  print('this is mine')
===
# 无歧义，无需上文
-  exit()
"""
# diff = """# 定位上文
#  def print():
# # 增加文本
# +  global count
# +  count += 1
# ~
#  if __name__ == '__main__':
# -  print(123)
# +  print('welcome to my world')
# +  print('this is mine')
# -  exit()
# """

print(modify(orig, diff))

count = 0
def print():
  global count
  count += 1
  hello()

if __name__ == '__main__':
  print('welcome to my world')
  print('this is mine')
  print('well')


In [3]:
orig="""document.addEventListener('DOMContentLoaded', () => {
  const contentDiv = document.getElementById('content');
  
  // 获取当前活动标签页
  chrome.tabs.query({ active: true, currentWindow: true }, (tabs) => {
    const tab = tabs[0];
    if (!tab) {
      contentDiv.innerHTML = '<div class="error">无法获取当前页面</div>';
      return;
    }

    // 向content script发送消息获取媒体资源
    chrome.tabs.sendMessage(tab.id, { action: 'getMedia' }, (response) => {
      if (chrome.runtime.lastError) {
        contentDiv.innerHTML = '<div class="error">无法与页面通信，请刷新页面后重试。</div>';
        console.error(chrome.runtime.lastError);
        return;
      }

      if (!response) {
        contentDiv.innerHTML = '<div class="error">未检测到媒体资源。</div>';
        return;
      }

      renderMedia(response);
    });
  });

  function renderMedia(media) {
    let html = '';

    // 图片
    if (media.images && media.images.length > 0) {
      html += `<div class="section">
        <h3>📷 图片 (${media.images.length})</h3>
        <div class="resource-list">`;
      media.images.forEach(url => {
        html += renderItem(url, 'image');
      });
      html += `</div></div>`;
    }

    // 视频
    if (media.videos && media.videos.length > 0) {
      html += `<div class="section">
        <h3>🎬 视频 (${media.videos.length})</h3>
        <div class="resource-list">`;
      media.videos.forEach(url => {
        html += renderItem(url, 'video');
      });
      html += `</div></div>`;
    }

    // 音频
    if (media.audios && media.audios.length > 0) {
      html += `<div class="section">
        <h3>🎵 音频 (${media.audios.length})</h3>
        <div class="resource-list">`;
      media.audios.forEach(url => {
        html += renderItem(url, 'audio');
      });
      html += `</div></div>`;
    }

    if (!media.images.length && !media.videos.length && !media.audios.length) {
      html = '<div class="loading">未找到媒体资源。</div>';
    }

    contentDiv.innerHTML = html;

    // 绑定下载按钮事件
    document.querySelectorAll('.download-btn').forEach(btn => {
      btn.addEventListener('click', (e) => {
        const url = btn.getAttribute('data-url');
        const type = btn.getAttribute('data-type');
        downloadMedia(url, type);
      });
    });
  }

  function renderItem(url, type) {
    // 提取文件名显示简短部分
    let filename = '';
    try {
      const urlObj = new URL(url);
      let pathname = urlObj.pathname;
      filename = pathname.substring(pathname.lastIndexOf('/') + 1);
      if (!filename) filename = 'file';
    } catch (e) {
      filename = 'file';
    }
    // 截断显示
    const shortUrl = filename.length > 30 ? filename.substring(0, 27) + '...' : filename;
    return `<div class="resource-item">
      <div class="resource-url" title="${url}">${shortUrl}</div>
      <button class="download-btn" data-url="${url}" data-type="${type}">下载</button>
    </div>`;
  }

  function downloadMedia(url, type) {
    // 生成建议文件名
    let filename = '';
    try {
      const urlObj = new URL(url);
      let pathname = urlObj.pathname;
      filename = pathname.substring(pathname.lastIndexOf('/') + 1);
      if (!filename) {
        filename = `download_${Date.now()}`;
      }
    } catch (e) {
      filename = `download_${Date.now()}`;
    }

    // 向background发送下载请求
    chrome.runtime.sendMessage({ action: 'download', url: url, filename: filename }, (response) => {
      if (response && response.success) {
        console.log('下载已开始');
      } else {
        alert('下载失败: ' + (response?.error || '未知错误'));
      }
    });
  }
});
"""
diff=""" document.addEventListener('DOMContentLoaded', () => {
   const contentDiv = document.getElementById('content');
+  let currentTab = null; // 保存当前标签页信息
  
   // 获取当前活动标签页
   chrome.tabs.query({ active: true, currentWindow: true }, (tabs) => {
     const tab = tabs[0];
     if (!tab) {
       contentDiv.innerHTML = '<div class="error">无法获取当前页面</div>';
       return;
     }
+    currentTab = tab; // 保存以便下载时使用
  
     // 向content script发送消息获取媒体资源
     chrome.tabs.sendMessage(tab.id, { action: 'getMedia' }, (response) => {
       if (chrome.runtime.lastError) {
         contentDiv.innerHTML = '<div class="error">无法与页面通信，请刷新页面后重试。</div>';
         console.error(chrome.runtime.lastError);
         return;
       }
  
       if (!response) {
         contentDiv.innerHTML = '<div class="error">未检测到媒体资源。</div>';
         return;
       }
  
       renderMedia(response);
     });
   });
  
   function renderMedia(media) {
     // ... (中间代码保持不变) ...
  
     // 绑定下载按钮事件
     document.querySelectorAll('.download-btn').forEach(btn => {
       btn.addEventListener('click', (e) => {
         const url = btn.getAttribute('data-url');
         const type = btn.getAttribute('data-type');
-        downloadMedia(url, type);
+        downloadMedia(url, type, currentTab);
       });
     });
   }
  
-  function downloadMedia(url, type) {
-    // 生成建议文件名
-    let filename = '';
-    try {
-      const urlObj = new URL(url);
-      let pathname = urlObj.pathname;
-      filename = pathname.substring(pathname.lastIndexOf('/') + 1);
-      if (!filename) {
-        filename = `download_${Date.now()}`;
-      }
-    } catch (e) {
-      filename = `download_${Date.now()}`;
-    }
+  // 生成格式化文件名: 日期 - 标题 - 域名.扩展名
+  function generateFilename(mediaUrl, pageTitle, pageUrl) {
+    // 1. 日期: YYYY-MM-DD
+    const now = new Date();
+    const year = now.getFullYear();
+    const month = String(now.getMonth() + 1).padStart(2, '0');
+    const day = String(now.getDate()).padStart(2, '0');
+    const datePart = `${year}-${month}-${day}`;
+    
+    // 2. 标题: 清理非法字符，过长则截断（可选，这里保留原样但替换非法字符）
+    let titlePart = pageTitle || '无标题';
+    titlePart = titlePart.replace(/[/\\:*?"<>|]/g, '_').trim();
+    if (titlePart.length > 50) titlePart = titlePart.substring(0, 50);
+    
+    // 3. 网址: 提取域名（hostname），清理非法字符（一般安全，但以防万一）
+    let domainPart = '';
+    try {
+      const urlObj = new URL(pageUrl);
+      domainPart = urlObj.hostname;
+    } catch (e) {
+      domainPart = 'unknown';
+    }
+    domainPart = domainPart.replace(/[/\\:*?"<>|]/g, '_');
+    
+    // 4. 扩展名: 从媒体URL中提取
+    let ext = '';
+    try {
+      const urlObj = new URL(mediaUrl);
+      const pathname = urlObj.pathname;
+      const lastDot = pathname.lastIndexOf('.');
+      if (lastDot !== -1 && lastDot < pathname.length - 1) {
+        ext = pathname.substring(lastDot + 1).split('?')[0]; // 去掉查询参数
+        ext = ext.toLowerCase();
+      }
+    } catch (e) {
+      ext = '';
+    }
+    if (!ext) ext = 'bin'; // 默认扩展名
+    
+    // 5. 拼接并清理文件名中的多余空格
+    let filename = `${datePart} - ${titlePart} - ${domainPart}.${ext}`;
+    // 替换多个空格为单个空格，并去除首尾空格
+    filename = filename.replace(/\s+/g, ' ').trim();
+    return filename;
+  }
+  
+  function downloadMedia(url, type, tab) {
+    if (!tab) {
+      alert('无法获取页面信息');
+      return;
+    }
+    const filename = generateFilename(url, tab.title, tab.url);
  
     // 向background发送下载请求
     chrome.runtime.sendMessage({ action: 'download', url: url, filename: filename }, (response) => {
       if (response && response.success) {
         console.log('下载已开始');
       } else {
         alert('下载失败: ' + (response?.error || '未知错误'));
       }
     });
   }
 });
"""

print(modify(orig, diff))

document.addEventListener('DOMContentLoaded', () => {
  const contentDiv = document.getElementById('content');

  // 获取当前活动标签页
  chrome.tabs.query({ active: true, currentWindow: true }, (tabs) => {
    const tab = tabs[0];
    if (!tab) {
      contentDiv.innerHTML = '<div class="error">无法获取当前页面</div>';
      return;
    }

    // 向content script发送消息获取媒体资源
    chrome.tabs.sendMessage(tab.id, { action: 'getMedia' }, (response) => {
      if (chrome.runtime.lastError) {
        contentDiv.innerHTML = '<div class="error">无法与页面通信，请刷新页面后重试。</div>';
        console.error(chrome.runtime.lastError);
        return;
      }

      if (!response) {
        contentDiv.innerHTML = '<div class="error">未检测到媒体资源。</div>';
        return;
      }

      renderMedia(response);
    });
  });

  function renderMedia(media) {
    let html = '';

    // 图片
    if (media.images && media.images.length > 0) {
      html += `<div class="section">
        <h3>📷 图片 (${media.images.length})</h3>
        <div class

<>:224: SyntaxWarning: invalid escape sequence '\s'
<>:224: SyntaxWarning: invalid escape sequence '\s'
C:\Users\Administrator\AppData\Local\Temp\ipykernel_17172\2225397440.py:224: SyntaxWarning: invalid escape sequence '\s'
  +    filename = filename.replace(/\s+/g, ' ').trim();
